# Il percettrone e la sua regola di apprendimento

Il codice del capitolo [«Il percettrone e la sua regola di apprendimento»](https://book.paithon.it/main/RetiNeurali/percettrone.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Il percettrone e la sua regola di apprendimento

[Leggi la pagina](https://book.paithon.it/main/RetiNeurali/percettrone.html)


### Imparare dagli errori: la regola del percettrone


In [ ]:
import numpy as np

def gradino(z):
    return np.where(z >= 0, 1, 0)          # decisione binaria 0/1

def addestra(X, y, eta=0.1, epoche=10):
    w = np.zeros(X.shape[1])               # pesi iniziali a zero
    b = 0.0
    for _ in range(epoche):
        for xi, target in zip(X, y):
            # la chiocciola @ è la somma pesata: w1*x1 + w2*x2 + ...
            pred = gradino(w @ xi + b)
            errore = target - pred
            w += eta * errore * xi          # aggiorna i pesi
            b += eta * errore               # aggiorna il bias
    return w, b

# La porta logica AND (vale 1 solo se entrambi gli ingressi valgono 1):
# i suoi quattro casi si separano con una retta
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])
w, b = addestra(X, y_and)
print(gradino(X @ w + b))                   # ha imparato la AND

## Funzioni di attivazione

[Leggi la pagina](https://book.paithon.it/main/RetiNeurali/funzioni-attivazione.html)


### In pratica, con NumPy


In [ ]:
import numpy as np

def sigmoide(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)                      # già in NumPy

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)   # pendenza alpha sui negativi

def softmax(z):
    z = z - np.max(z, axis=-1, keepdims=True)   # stabilità numerica
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)    # una riga per esempio

print(softmax(np.array([2.0, 1.0, 0.1])))

## Da dove viene la loss

[Leggi la pagina](https://book.paithon.it/main/RetiNeurali/da-dove-viene-la-loss.html)


### In pratica, con NumPy


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Duecento misure. La media vera e' una retta; il rumore no: nella meta'
# destra e' sei volte piu' largo che nella sinistra.
n = 200
x = np.sort(rng.uniform(0, 1, n))
sigma_vero = np.where(x < 0.5, 0.1, 0.6)
y = 2 * x + 1 + rng.normal(0, sigma_vero)

# Un solo centro per tutti e due i modelli: la retta ai minimi quadrati.
a, b = np.polyfit(x, y, 1)
residui = y - (a * x + b)

def nll(residui, sigma):
    "log-verosimiglianza gaussiana cambiata di segno, sommata sugli esempi"
    return np.sum(0.5 * np.log(2 * np.pi * sigma**2)
                  + residui**2 / (2 * sigma**2))

# 1. A larghezza fissa la NLL e' l'errore quadratico piu' una costante.
sse = np.sum(residui**2)
print(f"NLL(sigma=1) = {nll(residui, 1.0):.3f}")
print(f"0,5 * SSE    = {0.5 * sse:.3f}   differenza = "
      f"{nll(residui, 1.0) - 0.5 * sse:.3f}")
print(f"n/2 * log(2 pi) = {n / 2 * np.log(2 * np.pi):.3f}")

# 2. Se la larghezza la sceglie la verosimiglianza, il minimo cade sullo
#    scarto quadratico medio: sia stringere sia allargare costa.
rmse = np.sqrt(np.mean(residui**2))
print(f"\nscarto quadratico medio: {rmse:.3f}")
for s in [0.5 * rmse, rmse, 2 * rmse, 10 * rmse]:
    print(f"  sigma = {s:.3f}  ->  NLL = {nll(residui, s):.1f}")

# 3. Stesso centro, stesso errore quadratico, due modi di dichiarare
#    l'incertezza: uno solo per tutti, oppure uno per meta'.
sinistra = x < 0.5
s_sx = np.sqrt(np.mean(residui[sinistra]**2))
s_dx = np.sqrt(np.mean(residui[~sinistra]**2))
print(f"\nerrore quadratico medio: {np.mean(residui**2):.4f} (uguale)")
print(f"una larghezza sola:  sigma = {rmse:.3f}          "
      f"NLL = {nll(residui, rmse):.1f}")
etero = nll(residui[sinistra], s_sx) + nll(residui[~sinistra], s_dx)
print(f"una per meta':       sigma = {s_sx:.3f} e {s_dx:.3f}  NLL = {etero:.1f}")

## Backpropagation: come impara una rete

[Leggi la pagina](https://book.paithon.it/main/RetiNeurali/backpropagation.html)


### In pratica, con PyTorch


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

import torch
from torch import nn, optim

model = nn.Sequential(
    nn.Linear(784, 64), nn.ReLU(),   # strato nascosto
    nn.Linear(64, 10),               # uscita: un punteggio per classe
)

criterion = nn.CrossEntropyLoss()                   # la loss
optimizer = optim.SGD(model.parameters(), lr=0.01)  # discesa del gradiente

for epoca in range(20):
    for X_batch, y_batch in train_loader:  # mini-batch di 32 esempi
        y_pred = model(X_batch)            # forward: la previsione
        loss = criterion(y_pred, y_batch)  # quanto abbiamo sbagliato
        optimizer.zero_grad()              # azzera i gradienti vecchi
        loss.backward()                    # backpropagation automatica
        optimizer.step()                   # aggiornamento dei pesi
```
